In [ ]:
# install dependencies
# pip install spacy
# pip install pandas
# pip install emoji
# pip install tqdm pandarallel

In [ ]:
#Ensure correct pip version - required for spacy
#pip install numpy==1.23.0
#!python -m spacy download en_core_web_sm

### load data

In [ ]:
import pandas as pd

In [2]:
#load from csv
data = pd.read_csv(r'data csv\yoda-full.csv')

In [4]:
data.head()

,date,content,lang,hashtags
0,2019-11-30 23:59:58+00:00,Cant wait until tomorrow night's episode @Rick...,en,NaN
1,2019-11-30 23:59:54+00:00,@phut_phut_sefka Watched the first 2 episodes ...,en,NaN
2,2019-11-30 23:59:54+00:00,@NewYorkRedBulls @Eyes_N_Heart @starwars @them...,en,NaN
3,2019-11-30 23:59:49+00:00,Viendo la serie del baby yoda que es aún mejor...,es,NaN
4,2019-11-30 23:59:45+00:00,Esta noche tengo plan 🍷✨\n#TheMandalorian \nht...,es,['TheMandalorian']


In [5]:
data.shape

(882314, 4)

In [ ]:
data.drop(data.columns[0], axis = 1, inplace = True)
data

,date,content,lang,hashtags
0,2021-01-31 23:59:59+00:00,@Lord_Bridgerton // I was thinking of having h...,en,NaN
1,2021-01-31 23:59:52+00:00,@hessaxgastina terminar bridgerton jaja,es,NaN
2,2021-01-31 23:59:49+00:00,@bridgerton my teenage son said he will scream...,en,"['Bridgerton', 'bridgertonnetflix', 'Bridgerto..."
3,2021-01-31 23:59:38+00:00,@jbunzie @bridgerton I shall write to the Quee...,en,NaN
4,2021-01-31 23:59:30+00:00,"«El amor, duque, lo conquista todo».\n#Bridgerton",es,['Bridgerton']
...,...,...,...,...
509227,2020-12-01 00:02:41+00:00,Que saudade que me deu do evento de Bridgerton 😥💛,pt,NaN
509228,2020-12-01 00:02:25+00:00,@_Bridgerton meu sonho ser adorável kkkkkkkk,pt,NaN
509229,2020-12-01 00:02:24+00:00,Não aguento mais esperar por Bridgerton. Daqui...,pt,NaN
509230,2020-12-01 00:00:25+00:00,simplesmente não aguento mais esperar pelo lan...,pt,NaN


In [ ]:
#check languages in dataset
langs = {}

for i in data['lang']:
    if i in langs:
        langs[i]+= 1
    else:
        langs[i] =1

langs

{'en': 686833,
 'es': 82761,
 'und': 27376,
 'pl': 1145,
 'pt': 26042,
 'tr': 2619,
 'fr': 23693,
 'it': 6346,
 'ja': 2171,
 'ru': 1340,
 'in': 2876,
 'tl': 1866,
 'el': 102,
 'eu': 1065,
 'ro': 748,
 'et': 399,
 'nl': 1548,
 'no': 141,
 'sv': 533,
 'sl': 369,
 'ht': 451,
 'hu': 541,
 'ca': 1310,
 'de': 5471,
 'ar': 2364,
 'cs': 228,
 'th': 440,
 'fi': 385,
 'bg': 57,
 'cy': 197,
 'da': 189,
 'lt': 102,
 'zh': 97,
 'is': 54,
 'iw': 36,
 'fa': 147,
 'lv': 75,
 'sr': 21,
 'ur': 1,
 'hi': 45,
 'ne': 4,
 'ko': 39,
 'uk': 48,
 'vi': 27,
 'si': 4,
 'ta': 3,
 'ml': 4,
 'bn': 1}

In [8]:
data = data[data['lang'] == 'en']
data.head()

,date,content,lang,hashtags
0,2019-11-30 23:59:58+00:00,Cant wait until tomorrow night's episode @Rick...,en,NaN
1,2019-11-30 23:59:54+00:00,@phut_phut_sefka Watched the first 2 episodes ...,en,NaN
2,2019-11-30 23:59:54+00:00,@NewYorkRedBulls @Eyes_N_Heart @starwars @them...,en,NaN
5,2019-11-30 23:59:43+00:00,but what if Mando takes off his helmet for bab...,en,['TheMandalorian']
6,2019-11-30 23:59:38+00:00,Me: Do ya wanna watch The Moldovian... Mongoli...,en,NaN


In [10]:
data.shape

(686833, 4)

### clean text

In [ ]:
import spacy
import en_core_web_sm

nlp = spacy.load("en_core_web_sm")

In [11]:
def clean_text(text):    
     
    import emoji, regex as re

    text.lower()
    text = re.sub(r'\bhttp\S*\b', '', text) #clean links
    text = emoji.replace_emoji(text, replace='') #clean emojis
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text) #clean special n punctuation

    return text

In [ ]:
from pandarallel import pandarallel
from tqdm import tqdm
pandarallel.initialize(progress_bar=True)
#tqdm.pandas(desc='preprocessing')

docs = data['content'].to_frame()

docs['content'] = docs['content'].parallel_apply(clean_text)

content, tokens, strings = [], [], [] #create lists for df columns

for doc in tqdm(nlp.pipe(docs['content'],batch_size=2000, n_process=4), desc="spaCy processing"): #[LOOK HERE] modify batch size and num processes for your needs
    lemma_list = [token.lemma_.lower() for token in doc if not token.is_stop]
    lemma_string = ' '.join(lemma_list).strip()

    content.append(doc.text.strip())
    tokens.append(lemma_list)
    strings.append(lemma_string)

docs = pd.DataFrame({"content": content, "tokens": tokens, "strings": strings}) #push processed text to df 

In [ ]:
docs.head()

,content,tokens,strings
0,Cant wait until tomorrow nights episode Rickan...,"[not, wait, tomorrow, night, episode, rickandm...",not wait tomorrow night episode rickandmorty t...
1,phutphutsefka Watched the first 2 episodes of ...,"[phutphutsefka, watch, 2, episode, mandalorian...",phutphutsefka watch 2 episode mandalorian far ...
2,NewYorkRedBulls EyesNHeart starwars themandalo...,"[newyorkredbull, eyesnheart, starwar, themanda...",newyorkredbull eyesnheart starwar themandalori...
3,Esta noche tengo plan \nTheMandalorian,"[esta, noche, tengo, plan, \n, themandalorian]",esta noche tengo plan \n themandalorian
4,but what if Mando takes off his helmet for bab...,"[mando, take, helmet, baby, yoda, , themandal...",mando take helmet baby yoda themandalorian


In [ ]:
docs.shape

(686833, 3)

In [ ]:
docs.to_csv('yoda_tokens.csv') #push results to csv